# Ligamento · Calibración R-BANDA — escalón **E-b**

Completa la escalera de `ENMIENDA_E-005_RBANDA_20260805.md` §3.1. **E-a ya corrió**
(2026-08-06, CPU, 87 min): cerró la bisección, cero cargas en la banda, `decidir()` devolvió
**R4 — decisión suspendida**. Falta el segundo escalón.

> **Por qué esto y no una enmienda.** R3 («frontera inalcanzable», la rama que dejaría a P2.2 sin
> correr) exige *«bisección cerrada en **ambos** escalones»*. E-b nunca corrió, así que R3 **no está
> disponible**, y R4 manda retomar la calibración. El texto congelado cierra con: *«Leer R4 como R3
> sería convertir una restricción de crédito en un hallazgo»* — y E-b no corrió por crédito (no
> entraba en las 3 h de la estación de trabajo). Ver `DICTAMEN_20260806.md`.

**Presupuesto: ya está congelado, no hace falta enmienda.** §4 reparte 6 h en dos mitades
declaradas —hasta 3 h para E-a, las 3 h restantes para E-b—. E-a consumió 87 min: la mitad de E-b
sigue íntegra.

**Hardware: T4 no es obligatorio acá.** §4, textual: *«Hardware indistinto, declarado. A diferencia
de las campañas, la calibración **no** exige Tesla T4 ni homogeneidad de hardware: no emite
veredictos»*. La homogeneidad sigue rigiendo **dentro de** E2–E4. Por eso el gate de abajo
**declara** la GPU en vez de bloquear.

**Qué NO es esto:** range-finding, sin veredictos. Su único producto es la elección de régimen.
Ninguna celda de acá confirma ni falsa ninguna predicción.

**Orden de uso:** las celdas 1→5 son segundos. **La 6 (bench) decide si conviene seguir** — si dice
que no entra, no sigas. La 7 es el control de numérica, la 8 la corrida larga, la 9 el diagnóstico
del corte (no la saltees) y la 10 dice cómo bajar el resultado.

In [ ]:
# 1) GATE DE HARDWARE — declara, no bloquea.
#
# A diferencia del notebook de E1 (que EXIGE T4 porque sus 19 corridas versionadas son de T4 y
# mezclar backends meteria el hardware adentro de la comparacion), la calibracion tiene hardware
# declarado-pero-indistinto por E-005 §4: no emite veredictos. Lo unico obligatorio es DEJAR
# ASENTADO en que corrio, para que el expediente lo tenga.
import subprocess, sys, os, datetime

g = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                    '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU asignada:', g or 'NINGUNA (correria en CPU)')
if not g:
    print('\n  Sin GPU esto NO entra en el tope de 3 h (en la estacion de trabajo son ~6 h).')
    print('  Entorno de ejecucion -> Cambiar tipo de entorno -> GPU, y volve a correr esta celda.')
HW = {'gpu': g or 'cpu', 'cuando': datetime.datetime.now().isoformat(timespec='seconds')}
print('\nquedara asentado como:', HW)

In [ ]:
# 2) Dependencias
!pip -q install optax >/dev/null
import jax; print('jax', jax.__version__, '| devices:', jax.devices())
print('plataforma:', jax.devices()[0].platform)

In [ ]:
# 3) Codigo pre-registrado (repo publico)
![ -d telar-ligamento ] && (cd telar-ligamento && git pull -q) || git clone -q https://github.com/SperanzaMax/telar-ligamento
%cd /content/telar-ligamento
!git log --oneline -1

In [ ]:
# 4) Drive: los checkpoints sobreviven al corte de sesion.
#
# IMPORTANTE: train_resumable REANUDA desde el .pkl si existe. Si el Drive pierde el archivo,
# el runner reentrena desde cero y tira el presupuesto (paso el 2026-08-01 con softmax s3-s7).
from google.colab import drive
drive.mount('/content/drive')
import os
CALIB = '/content/drive/MyDrive/ligamento_calibracion'
CKPT  = os.path.join(CALIB, 'ckpt')
os.makedirs(CKPT, exist_ok=True)
print('salida      ->', CALIB)
print('checkpoints ->', CKPT)
print('\ncontenido actual:')
for f in sorted(os.listdir(CKPT)) or ['(vacio)']:
    p = os.path.join(CKPT, f)
    print('  ', f, (os.path.getsize(p) // 1024 if os.path.isfile(p) else ''), 'KB')

In [ ]:
# 5) GATE DE PRE-REGISTRO. Bloquea si algun artefacto congelado no coincide con su hash.
!python experimentos/verificar_anclas.py --requiere E-005

## 6 · Cuánto cuesta en ESTA GPU — se mide, no se estima

El bench del 2026-08-05 **subestimó el costo un 44 %** (1,815 s/paso proyectados contra 2,60
medidos), y ésa fue la causa raíz de que E-b no entrara en el tope y de que la calibración volviera
R4 (`desviaciones.md` D-006(b)). No se repite el error: se mide acá y se decide con el número real.

**Referencias, todas medidas** (estación de trabajo, batch 64, `softmax`):

| escalón | T efectivo (4·L+2) | s/paso CPU | origen |
|---|---|---|---|
| E-a (NK 256, L 128) | 514 | **2,60** | corrida real del 2026-08-06 |
| E-a (NK 256, L 128) | 514 | 3,19 | este mismo bench → **sobreestima 23 %** |
| E-b (NK 512, L 256) | 1026 | **11,36** | este mismo bench (≈3,6× E-a) |

Que el bench sobreestime es el lado seguro, y da la corrección: E-b en esta CPU debería costar
**≈9,3 s/paso reales**. La T4 tiene que ganarle **al menos 3×** para que entre.

E-a **convergió a 1000 pasos** de un tope de 2500 — si E-b se le parece, entra; si necesita el tope
entero, no. La celda mide los dos escalones: E-a sirve de control (sabemos cuánto tiene que dar) y
de ahí sale el factor GPU/CPU real de esta máquina.

In [ ]:
# 6) Costo real en ESTA GPU. ~4-6 min. Solo cronometra el paso: no toca ninguna metrica de tarea.
#
#    Corre PRIMERO E-a a proposito: de ese escalon tenemos el numero REAL de la corrida del
#    2026-08-06 (2,60 s/paso en CPU), asi que hace de doble control — valida el bench y da el
#    factor GPU/CPU de esta maquina, que es lo que decide si E-b entra en el tope.
!python experimentos/E1/bench_calibracion.py --escalon E-a --pasos 20
print('\n' + '=' * 78 + '\n')
!python experimentos/E1/bench_calibracion.py --escalon E-b --pasos 20

## 7 · Control de numérica — barato, sin reentrenar

E-a se entrenó en CPU y E-b va a correr en GPU. Antes de darle crédito a cualquier medida de E-b
conviene saber que el pipeline de **evaluación** da lo mismo en los dos backends.

El control no reentrena nada: toma los **checkpoints de E-a que ya existen**, los evalúa acá y
compara contra lo que dieron en CPU (`acc@1 = 1,0000` en L≤96 y `0,9999` en L=128). Son minutos.

**Requisito:** subir `calib_E-a_s0.pkl` y `calib_E-a_s1.pkl` (2,4 MB c/u, están en
`resultados/calibracion/ckpt/` de la máquina local — no van al repo porque `*.pkl` está en
`.gitignore`) a la carpeta `ligamento_calibracion/ckpt` del Drive. **Si no están, la celda lo dice y
se puede seguir igual**: el control es una salvaguarda, no un gate del protocolo.

In [ ]:
# 7) Control de numerica CPU-vs-GPU sobre los checkpoints de E-a. No reentrena.
import os, pickle, sys
sys.path.insert(0, 'src'); sys.path.insert(0, 'experimentos/E1')

ESPERADO_CPU = {8: 1.0000, 16: 1.0000, 32: 1.0000, 64: 1.0000, 96: 1.0000, 128: 0.9999}
TOL = 0.01   # 1 punto: holgado a proposito, buscamos roturas de numerica, no ruido

faltan = [f'calib_E-a_s{s}.pkl' for s in (0, 1)
          if not os.path.exists(os.path.join(CKPT, f'calib_E-a_s{s}.pkl'))]
if faltan:
    print('  No estan en el Drive:', ', '.join(faltan))
    print('   Subilos a', CKPT, 'desde resultados/calibracion/ckpt/ de la maquina local.')
    print('   Se puede seguir sin esto: es salvaguarda, no gate. Solo perdes el control.')
else:
    from calibrar_rbanda import ESCALONES, medir
    from datos import hacer_vocab
    e_a = ESCALONES[0]
    voc = hacer_vocab(e_a['NK'], 64)
    peor = 0.0
    for s in (0, 1):
        with open(os.path.join(CKPT, f'calib_E-a_s{s}.pkl'), 'rb') as fh:
            params = pickle.load(fh)['params']
        got = medir(params, voc, e_a['grilla'])
        print(f'\n  semilla {s}:')
        for L in e_a['grilla']:
            d = abs(got[L] - ESPERADO_CPU[L]); peor = max(peor, d)
            flag = 'ok' if d <= TOL else '  DIFIERE'
            print(f'    L={L:>4}  GPU={got[L]:.4f}  CPU={ESPERADO_CPU[L]:.4f}  |d|={d:.4f}  {flag}')
    print(f'\n  peor diferencia: {peor:.4f} (tolerancia {TOL})')
    print('  CONTROL OK — la evaluacion es equivalente entre backends.' if peor <= TOL else
          '  CONTROL FALLA — no sigas: hay un problema de numerica que contaminaria E-b.')

In [ ]:
# 8a) (opcional) Aviso por Telegram. El token va SOLO por env, nunca al repo ni al notebook guardado.
import os
os.environ['TG_TOKEN'] = ''        # <- pegar token aca si se quiere aviso; vacio = sin avisos
os.environ['TG_CHAT']  = '7985522502'
print('avisos Telegram:', 'ON' if os.environ['TG_TOKEN'] else 'OFF')

## 8 · La corrida

Un solo comando, con los parámetros congelados. **No se toca ninguna constante**: banda
[0,50 · 0,80], k=1, `softmax`, semillas (0, 1), parada por convergencia con tope 2500, tope 180 min.

Es **reanudable**: si Colab corta, volvé a correr esta misma celda y `train_resumable` sigue desde
el checkpoint del Drive. Por eso conviene tener el Drive montado y verificado (celda 4).

**Memoria:** con `T=1026` y batch 64 la atención pide del orden de 1 GB por matriz en float32.
En una T4 (16 GB) entra, pero si aparece un OOM **no bajes el batch por tu cuenta** —cambiaría el
régimen de entrenamiento respecto de E-a y rompería la comparabilidad de la escalera—: pará y
registralo como desviación.

Las tres salidas posibles, todas legítimas y ninguna deja el expediente peor que hoy:

| si… | rama | qué significa |
|---|---|---|
| E-b cumple la banda | **R2** | Régimen de E2–E4 = E-b (NK 512, L 256). P2.1/P2.2/P2.3 corren con umbrales intactos. **Destrabado.** |
| E-b no cumple, bisección cerrada | **R3** | Frontera inalcanzable, ahora **medida** con la escalera completa. P2.2 se registra NO CORRIDA, nunca confirmada. |
| se agota el tope con la bisección abierta | **R4** | Se retoma desde donde quedó. No se reinterpreta. |

**No interpretes la rama a mano: la emite `decidir()` con los dos escalones corridos.**

In [ ]:
# 8b) LA CORRIDA. Reanudable: si Colab corta, volve a correr esta celda tal cual.
!python experimentos/E1/calibrar_rbanda.py \
    --escalon E-b \
    --salida "{CALIB}" \
    --ckpt "{CKPT}" \
    2>&1 | tee -a "{CALIB}/corrida_E-b.log"

## 9 · Diagnóstico del corte — **no saltear esta celda**

La calibración corta por convergencia con **ventana 500 / tol 0,005** (criterio D-004). El
2026-08-06 se descubrió que ese criterio **se dispara adentro de mesetas**: en `d=8` una meseta de
**750 pasos** —más larga que la ventana— fue leída como convergencia, y el modelo después despegó
(`desviaciones.md` D-006(c)).

**Por qué importa acá, y en qué dirección.** E-a no estuvo expuesto: cortó a 1000 pasos con
`acc@1 = 1,0000`, y un modelo en el techo no tiene a dónde despegar. **E-b es el escalón difícil**,
y ahí el riesgo es real y perverso: si corta dentro de una meseta, devuelve una accuracy
artificialmente baja que podría **caer en la banda por subentrenamiento en vez de por capacidad**, y
el runner elegiría E-b como régimen de E2–E4 por el motivo equivocado.

El criterio congelado **no se toca** —eso exigiría enmienda—. Esta celda es diagnóstico: **no
cambia el veredicto**, lo rotula. Si dispara, se registra como desviación y se decide con
presupuesto declarado.

In [ ]:
# 9) Diagnostico: el corte, cayo dentro de una meseta? NO cambia el veredicto, lo rotula.
import os, pickle

for s in (0, 1):
    ruta = os.path.join(CKPT, f'calib_E-b_s{s}.pkl')
    if not os.path.exists(ruta):
        print(f'  falta {ruta} (la corrida no llego a esta semilla)'); continue
    with open(ruta, 'rb') as fh:
        d = pickle.load(fh)
    vh = d.get('val_hist', [])
    if not vh:
        print(f'  semilla {s}: sin val_hist'); continue
    pasos = [h['step'] for h in vh]; vals = [h['val_acc'] for h in vh]
    print(f'\n=== semilla {s} · corte en {pasos[-1]} pasos ===')
    print('  curva:', '  '.join(f'{p}:{v:.3f}' for p, v in zip(pasos, vals)))
    pico = max(vals); i_pico = vals.index(pico)
    print(f'  pico {pico:.4f} @ {pasos[i_pico]}   final {vals[-1]:.4f}')

    banderas = []
    # (i) corto LEJOS del techo -> el patron meseta-y-despegue es posible
    if vals[-1] < 0.95:
        banderas.append(f'corto lejos del techo (val {vals[-1]:.3f} < 0,95): meseta-y-despegue POSIBLE')
    else:
        print('  ok: corto en el techo — sin lugar donde despegar, como E-a')
    # (ii) venia SUBIENDO antes del corte -> sospecha fuerte de meseta
    if len(vals) >= 3 and (vals[-1] - vals[-3]) > 0.02:
        banderas.append(f'venia subiendo fuerte (+{vals[-1]-vals[-3]:.3f} en los ultimos 1000 pasos)')
    # (iii) divergencia (el flag del dictamen, punto 2)
    if vals[-1] < pico - 0.05:
        banderas.append(f'DIVERGENCIA: final {vals[-1]:.3f} < pico {pico:.3f} — run INESTABLE')

    if banderas:
        print('  BANDERAS:')
        for b in banderas:
            print('   -', b)
        print('  -> registrar como desviacion antes de usar este numero para elegir regimen.')
    else:
        print('  sin banderas: el corte es creible.')

## 10 · Bajar el resultado al repo

1. Descargá de `MyDrive/ligamento_calibracion/`: **`calibracion_rbanda.json`** y
   **`corrida_E-b.log`**.
2. En la máquina local, ponelos en `resultados/calibracion/` (el `.json` se versiona; el `.log` no,
   por la regla `*.log` — si tiene números que hay que conservar, destilalos como se hizo con
   `RESUMEN_sondeo_d_20260806.json`).
3. Guardá también los `calib_E-b_s*.pkl` **fuera del repo** (`*.pkl` está en `.gitignore`): sirven
   para reanudar y para el control de numérica de la próxima.
4. Commiteá con la rama que emitió `decidir()` **en el título**, y sin interpretarla.

**Recordatorio final:** esto es range-finding. No emite veredictos, no confirma ni falsa ninguna
predicción, y su único producto es la elección de régimen para E2–E4.